<a href="https://colab.research.google.com/github/goitstudent123/llm-gen/blob/main/dz_final_has.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
# Chunk 1: Install dependencies
!pip -q install -U pandas openpyxl requests tqdm langchain langchain-openai
!pip -q check

In [63]:
# Chunk 2: Initial imports and OpenRouter setup
import os
import re
import json
import math
import time
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI


def read_colab_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_api_key = (
    read_colab_secret("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_API_KEY")
)

OPENROUTER_MODEL = (
    read_colab_secret("OPENROUTER_MODEL")
    or os.getenv("OPENROUTER_MODEL")
    or "openai/gpt-4.1-mini"
)

OPENROUTER_HTTP_REFERER = (
    read_colab_secret("OPENROUTER_HTTP_REFERER")
    or os.getenv("OPENROUTER_HTTP_REFERER")
    or "https://colab.research.google.com"
)

OPENROUTER_APP_TITLE = (
    read_colab_secret("OPENROUTER_APP_TITLE")
    or os.getenv("OPENROUTER_APP_TITLE")
    or "Colab Excel Enrichment"
)

llm = None

if openrouter_api_key:
    llm = ChatOpenAI(
        model=OPENROUTER_MODEL,
        api_key=openrouter_api_key,
        base_url=OPENROUTER_BASE_URL,
        temperature=0,
        default_headers={
            "HTTP-Referer": OPENROUTER_HTTP_REFERER,
            "X-OpenRouter-Title": OPENROUTER_APP_TITLE,
        },
    )
    print(f"OpenRouter model: {OPENROUTER_MODEL}")
else:
    print("OPENROUTER_API_KEY was not found. The deterministic planner will be used.")

OpenRouter model: openai/gpt-4.1-mini


In [64]:
# Chunk 3: Colab file upload and Google Sheets download helpers
def upload_excel_files() -> List[str]:
    try:
        from google.colab import files
        uploaded = files.upload()
        return list(uploaded.keys())
    except Exception as error:
        print(f"Upload is available only in Colab: {error}")
        return []


def download_result_file(file_path: str) -> None:
    try:
        from google.colab import files
        files.download(file_path)
    except Exception as error:
        print(f"Download is available only in Colab: {error}")


def extract_google_sheet_id(url: str) -> str:
    match = re.search(r"/spreadsheets/d/([a-zA-Z0-9-_]+)", url)
    if not match:
        raise ValueError("Could not extract Google Sheet ID from URL.")
    return match.group(1)


def extract_google_sheet_gid(url: str) -> str:
    match = re.search(r"gid=([0-9]+)", url)
    return match.group(1) if match else "0"


def download_google_sheet_as_xlsx(url: str, output_path: str) -> str:
    sheet_id = extract_google_sheet_id(url)
    gid = extract_google_sheet_gid(url)

    export_url = (
        f"https://docs.google.com/spreadsheets/d/{sheet_id}/export"
        f"?format=xlsx&gid={gid}"
    )

    response = requests.get(export_url, timeout=30)

    if response.status_code != 200:
        raise RuntimeError(
            f"Could not download Google Sheet. Status: {response.status_code}"
        )

    Path(output_path).write_bytes(response.content)
    return output_path

In [65]:
# Chunk 4: Create sample Excel files
def create_sample_excel_files() -> Tuple[str, str]:
    capitals_df = pd.DataFrame(
        [
            {
                "Capital_From": "Kyiv",
                "Country_From": "Ukraine",
                "Capital_To": "Paris",
                "Country_To": "France",
                "distance": None,
            },
            {
                "Capital_From": "London",
                "Country_From": "United Kingdom",
                "Capital_To": "Rome",
                "Country_To": "Italy",
                "distance": None,
            },
            {
                "Capital_From": "Paris",
                "Country_From": "France",
                "Capital_To": "Berlin",
                "Country_To": "Germany",
                "distance": None,
            },
            {
                "Capital_From": "Berlin",
                "Country_From": "Germany",
                "Capital_To": "Vienna",
                "Country_To": "Austria",
                "distance": None,
            },
            {
                "Capital_From": "Warsaw",
                "Country_From": "Poland",
                "Capital_To": "Kyiv",
                "Country_To": "Ukraine",
                "distance": None,
            },
        ]
    )

    mountains_df = pd.DataFrame(
        [
            {"Mountain": "Everest", "Country": "Nepal", "height": None},
            {"Mountain": "Mont Blanc", "Country": "France", "height": None},
            {"Mountain": "Denali", "Country": "USA", "height": None},
            {"Mountain": "Kilimanjaro", "Country": "Tanzania", "height": None},
        ]
    )

    capitals_path = "capitals.xlsx"
    mountains_path = "mountains.xlsx"

    capitals_df.to_excel(capitals_path, index=False)
    mountains_df.to_excel(mountains_path, index=False)

    return capitals_path, mountains_path


capitals_path, mountains_path = create_sample_excel_files()

print(capitals_path)
print(mountains_path)

capitals.xlsx
mountains.xlsx


In [66]:
# Chunk 5: Analyze task and build enrichment plan
def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def find_column_case_insensitive(columns: List[str], expected_name: str) -> Optional[str]:
    expected_lower = expected_name.lower()

    for column in columns:
        if column.lower() == expected_lower:
            return column

    return None


def deterministic_plan(df: pd.DataFrame, task_description: str) -> Dict[str, Any]:
    columns = list(df.columns)
    text = task_description.lower()

    distance_column = find_column_case_insensitive(columns, "distance")
    height_column = find_column_case_insensitive(columns, "height")

    if distance_column or "відстан" in text or "distance" in text:
        return {
            "task_type": "distance_between_places",
            "target_column": distance_column or "distance",
            "required_columns": [
                "Capital_From",
                "Country_From",
                "Capital_To",
                "Country_To",
            ],
            "from_place_columns": ["Capital_From", "Country_From"],
            "to_place_columns": ["Capital_To", "Country_To"],
            "place_context_suffix": "capital city",
            "value_unit": "km",
        }

    if height_column or "висот" in text or "height" in text:
        return {
            "task_type": "wikidata_quantity_property",
            "target_column": height_column or "height",
            "required_columns": [
                "Mountain",
                "Country",
            ],
            "entity_column": "Mountain",
            "context_columns": ["Country"],
            "entity_context_suffix": "mountain",
            "wikidata_property_id": "P2044",
            "value_unit": "m",
        }

    return {
        "task_type": "unsupported",
        "target_column": "",
        "required_columns": [],
        "value_unit": "",
    }


def normalize_enrichment_plan(
    parsed: Optional[Dict[str, Any]],
    fallback_plan: Dict[str, Any],
) -> Dict[str, Any]:
    if not parsed:
        return fallback_plan

    task_type = parsed.get("task_type")

    if task_type not in {
        "distance_between_places",
        "wikidata_quantity_property",
        "unsupported",
    }:
        return fallback_plan

    if task_type == "unsupported":
        return {
            "task_type": "unsupported",
            "target_column": "",
            "required_columns": [],
            "value_unit": "",
        }

    if not parsed.get("target_column"):
        return fallback_plan

    if not parsed.get("required_columns"):
        return fallback_plan

    if task_type == "distance_between_places":
        if not parsed.get("from_place_columns") or not parsed.get("to_place_columns"):
            return fallback_plan

        return {
            "task_type": "distance_between_places",
            "target_column": parsed["target_column"],
            "required_columns": parsed["required_columns"],
            "from_place_columns": parsed["from_place_columns"],
            "to_place_columns": parsed["to_place_columns"],
            "place_context_suffix": parsed.get("place_context_suffix", ""),
            "value_unit": parsed.get("value_unit", "km"),
        }

    if task_type == "wikidata_quantity_property":
        if not parsed.get("entity_column") or not parsed.get("wikidata_property_id"):
            return fallback_plan

        return {
            "task_type": "wikidata_quantity_property",
            "target_column": parsed["target_column"],
            "required_columns": parsed["required_columns"],
            "entity_column": parsed["entity_column"],
            "context_columns": parsed.get("context_columns", []),
            "entity_context_suffix": parsed.get("entity_context_suffix", ""),
            "wikidata_property_id": parsed["wikidata_property_id"],
            "value_unit": parsed.get("value_unit", ""),
        }

    return fallback_plan


def analyze_task(df: pd.DataFrame, task_description: str) -> Dict[str, Any]:
    fallback_plan = deterministic_plan(df, task_description)

    if llm is None:
        return fallback_plan

    sample_rows = df.head(5).fillna("").to_dict(orient="records")

    prompt = f"""
You analyze Excel enrichment tasks.

Return JSON only with this schema:
{{
  "task_type": "distance_between_places|wikidata_quantity_property|unsupported",
  "target_column": "column name to fill or create",
  "required_columns": ["column names needed from the file"],
  "from_place_columns": ["name column", "optional context column"],
  "to_place_columns": ["name column", "optional context column"],
  "place_context_suffix": "optional search hint, for example capital city",
  "entity_column": "entity name column",
  "context_columns": ["optional context columns"],
  "entity_context_suffix": "optional search hint, for example mountain",
  "wikidata_property_id": "Wikidata property id, for example P2044",
  "value_unit": "km|m|"
}}

Use "distance_between_places" when the task asks to calculate distance from coordinates.
Use "wikidata_quantity_property" when the task asks to fill a numeric value from Wikidata.

Task description:
{task_description}

Columns:
{list(df.columns)}

Sample rows:
{sample_rows}
""".strip()

    try:
        response = llm.invoke(prompt)
        parsed = extract_json_object(response.content)
        return normalize_enrichment_plan(parsed, fallback_plan)
    except Exception:
        return fallback_plan


def validate_required_columns(df: pd.DataFrame, required_columns: List[str]) -> None:
    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

In [67]:
# Chunk 6: Wikidata internet lookup helpers
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"

HTTP_HEADERS = {
    "User-Agent": "colab-excel-enrichment/1.0 educational project"
}


def request_json(
    url: str,
    params: Dict[str, Any],
    timeout: int = 30,
    attempts: int = 3,
) -> Dict[str, Any]:
    import time

    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=HTTP_HEADERS,
                timeout=timeout,
            )
            response.raise_for_status()
            return response.json()
        except Exception as error:
            last_error = error

            if attempt < attempts:
                time.sleep(attempt)

    raise last_error


def search_wikidata_entities(query: str, limit: int = 10) -> List[Dict[str, Any]]:
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "search": query,
        "limit": limit,
    }

    data = request_json(WIKIDATA_API_URL, params)
    return data.get("search", [])


def get_wikidata_entities(qids: List[str]) -> Dict[str, Any]:
    if not qids:
        return {}

    unique_qids = list(dict.fromkeys(qids))

    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": "|".join(unique_qids),
        "props": "claims|labels|descriptions",
        "languages": "en",
    }

    data = request_json(WIKIDATA_API_URL, params)
    return data.get("entities", {})


def sparql_escape(value: str) -> str:
    return value.replace("\\", "\\\\").replace('"', '\\"')


def sparql_label_values(labels: List[str]) -> str:
    return " ".join(
        f'"{sparql_escape(label)}"@en'
        for label in labels
        if label
    )


def query_wikidata_sparql(query: str, timeout: int = 30) -> Dict[str, Any]:
    params = {
        "query": query,
        "format": "json",
    }

    return request_json(
        WIKIDATA_SPARQL_URL,
        params,
        timeout=timeout,
        attempts=2,
    )


def get_claim_values(entity: Dict[str, Any], property_id: str) -> List[Any]:
    claims = entity.get("claims", {}).get(property_id, [])
    values = []

    for claim in claims:
        main_snak = claim.get("mainsnak", {})
        data_value = main_snak.get("datavalue", {})

        if "value" in data_value:
            values.append(data_value["value"])

    return values


def get_claim_value(entity: Dict[str, Any], property_id: str) -> Optional[Any]:
    values = get_claim_values(entity, property_id)

    if not values:
        return None

    return values[0]


def extract_wikibase_qids(entity: Dict[str, Any], property_id: str) -> List[str]:
    qids = []

    for value in get_claim_values(entity, property_id):
        if isinstance(value, dict) and value.get("entity-type") == "item":
            qid = value.get("id")

            if qid:
                qids.append(qid)

    return list(dict.fromkeys(qids))


def get_entity_labels(qids: List[str]) -> Dict[str, str]:
    entities = get_wikidata_entities(qids)
    labels = {}

    for qid, entity in entities.items():
        labels[qid] = entity_label(entity)

    return labels


def labels_for_claim(entity: Dict[str, Any], property_id: str) -> List[str]:
    qids = extract_wikibase_qids(entity, property_id)

    if not qids:
        return []

    labels_by_qid = get_entity_labels(qids)

    return [
        label
        for label in labels_by_qid.values()
        if label
    ]


def extract_coordinates(entity: Dict[str, Any]) -> Optional[Tuple[float, float]]:
    values = get_claim_values(entity, "P625")

    for value in values:
        if isinstance(value, dict) and "latitude" in value and "longitude" in value:
            return float(value["latitude"]), float(value["longitude"])

    return None


def extract_quantity_amount(
    entity: Dict[str, Any],
    property_id: str,
) -> Optional[int]:
    values = get_claim_values(entity, property_id)

    for value in values:
        if isinstance(value, dict) and "amount" in value:
            return int(round(float(value["amount"])))

    return None


def entity_label(entity: Dict[str, Any]) -> str:
    labels = entity.get("labels", {})
    english_label = labels.get("en", {})

    return english_label.get("value", "")


def entity_description(entity: Dict[str, Any]) -> str:
    descriptions = entity.get("descriptions", {})
    english_description = descriptions.get("en", {})

    return english_description.get("value", "")


def entity_url(qid: str) -> str:
    return f"https://www.wikidata.org/wiki/{qid}"


def qid_from_uri(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1]


def parse_wikidata_point(value: str) -> Optional[Tuple[float, float]]:
    match = re.match(r"Point\(([-\d.]+) ([-\d.]+)\)", value)

    if not match:
        return None

    longitude = float(match.group(1))
    latitude = float(match.group(2))

    return latitude, longitude

In [68]:
# Chunk 7: Generic enrichment lookup logic with detailed audit information
def clean_text(value: Any) -> str:
    if pd.isna(value):
        return ""

    return str(value).strip()


def build_search_text(*parts: Any) -> str:
    return " ".join(
        part
        for part in (clean_text(value) for value in parts)
        if part
    )


def normalize_for_match(value: Any) -> str:
    text = clean_text(value).lower()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def split_context_values(context: str) -> List[str]:
    raw_parts = re.split(r"[/,;|]+", clean_text(context))

    return [
        part.strip()
        for part in raw_parts
        if part.strip()
    ]


def unique_texts(values: List[str]) -> List[str]:
    result = []

    for value in values:
        if value and value not in result:
            result.append(value)

    return result


def generate_label_variants(entity_name: str, context_suffix: str = "") -> List[str]:
    base_name = clean_text(entity_name)
    variants = [base_name]

    if base_name.lower().startswith("mount "):
        variants.append(base_name[6:])

    if "mountain" in clean_text(context_suffix).lower():
        if not base_name.lower().startswith("mount "):
            variants.append(f"Mount {base_name}")

        if not re.search(r"\b(i|ii|iii|iv|v)\b$", base_name.lower()):
            variants.append(f"{base_name} I")

    return unique_texts(variants)


def matched_context_parts(text: str, context: str) -> List[str]:
    normalized_text = normalize_for_match(text)
    matches = []

    for part in split_context_values(context):
        normalized_part = normalize_for_match(part)

        if normalized_part and normalized_part in normalized_text:
            matches.append(part)

    return matches


def haversine_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    earth_radius_km = 6371.0088

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)

    value = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(lat1_rad)
        * math.cos(lat2_rad)
        * math.sin(delta_lon / 2) ** 2
    )

    central_angle = 2 * math.atan2(math.sqrt(value), math.sqrt(1 - value))
    return earth_radius_km * central_angle


def empty_lookup_result(
    value_key: str,
    input_name: str,
    context: str,
    reason: str,
    candidates: Optional[List[Dict[str, Any]]] = None,
) -> Dict[str, Any]:
    return {
        "ok": False,
        value_key: None,
        "source": "",
        "status": "not_found",
        "method": "",
        "qid": "",
        "label": "",
        "country": "",
        "reason": reason,
        "candidates": candidates or [],
        "input": input_name,
        "context": context,
    }


def candidate_log_entry(candidate: Dict[str, Any]) -> str:
    qid = candidate.get("qid", "")
    label = candidate.get("label", "")
    country = candidate.get("country", "")
    decision = candidate.get("decision", "")
    reason = candidate.get("reason", "")

    return f"{qid} {label} [{country}] -> {decision}: {reason}"


def select_place_candidate(
    candidates: List[Dict[str, Any]],
    place_name: str,
    context: str,
) -> Optional[Dict[str, Any]]:
    accepted_candidates = []

    for candidate in candidates:
        coordinates = candidate.get("coordinates")

        if not coordinates:
            candidate["decision"] = "rejected"
            candidate["reason"] = "candidate has no coordinates"
            candidate["score"] = 0
            continue

        candidate_text = build_search_text(
            candidate.get("label", ""),
            candidate.get("country", ""),
            candidate.get("description", ""),
            candidate.get("instance", ""),
        )

        context_matches = matched_context_parts(candidate_text, context)
        normalized_label = normalize_for_match(candidate.get("label", ""))
        normalized_name = normalize_for_match(place_name)

        label_exact = normalized_label == normalized_name
        label_contains_name = normalized_name and normalized_name in normalized_label

        if context and not context_matches:
            candidate["decision"] = "rejected"
            candidate["reason"] = "country/context does not match"
            candidate["score"] = 0
            continue

        score = 0

        if context_matches:
            score += 100

        if label_exact:
            score += 40
        elif label_contains_name:
            score += 15

        normalized_candidate_text = normalize_for_match(candidate_text)

        if "capital" in normalized_candidate_text:
            score += 10

        if "city" in normalized_candidate_text:
            score += 5

        candidate["decision"] = "accepted"
        candidate["reason"] = f"context matched: {', '.join(context_matches) if context_matches else 'not required'}"
        candidate["score"] = score
        accepted_candidates.append(candidate)

    if not accepted_candidates:
        return None

    accepted_candidates.sort(
        key=lambda item: item.get("score", 0),
        reverse=True,
    )

    selected = accepted_candidates[0]
    selected["decision"] = "selected"
    selected["reason"] = f"selected best matching place candidate; score={selected.get('score', 0)}"

    return selected


def select_quantity_candidate(
    candidates: List[Dict[str, Any]],
    entity_name: str,
    context: str,
    property_id: str,
) -> Optional[Dict[str, Any]]:
    accepted_candidates = []

    for candidate in candidates:
        value = candidate.get("value")

        if value is None:
            candidate["decision"] = "rejected"
            candidate["reason"] = "candidate has no numeric value"
            candidate["score"] = 0
            continue

        candidate_text = build_search_text(
            candidate.get("label", ""),
            candidate.get("country", ""),
            candidate.get("description", ""),
            candidate.get("instance", ""),
        )

        context_matches = matched_context_parts(candidate_text, context)
        normalized_label = normalize_for_match(candidate.get("label", ""))
        normalized_name = normalize_for_match(entity_name)

        label_exact = normalized_label == normalized_name
        label_starts_with_name = normalized_label.startswith(normalized_name)

        score = 0

        if context_matches:
            score += 100

        if label_exact:
            score += 40
        elif label_starts_with_name:
            score += 20

        normalized_candidate_text = normalize_for_match(candidate_text)

        if "mountain" in normalized_candidate_text or "peak" in normalized_candidate_text:
            score += 10

        if context and not context_matches:
            candidate["decision"] = "accepted_with_warning"
            candidate["reason"] = "no country/context match, accepted only as weak candidate"
        else:
            candidate["decision"] = "accepted"
            candidate["reason"] = f"context matched: {', '.join(context_matches) if context_matches else 'not required'}"

        candidate["score"] = score
        candidate["context_score"] = 1 if context_matches else 0
        candidate["value_for_sorting"] = value if isinstance(value, (int, float)) else -1

        accepted_candidates.append(candidate)

    if not accepted_candidates:
        return None

    if property_id == "P2044":
        accepted_candidates.sort(
            key=lambda item: (
                item.get("context_score", 0),
                item.get("value_for_sorting", -1),
                item.get("score", 0),
            ),
            reverse=True,
        )
    else:
        accepted_candidates.sort(
            key=lambda item: item.get("score", 0),
            reverse=True,
        )

    selected = accepted_candidates[0]
    selected["decision"] = "selected"

    if property_id == "P2044":
        selected["reason"] = (
            "selected best elevation candidate; "
            f"value={selected.get('value')}, score={selected.get('score', 0)}"
        )
    else:
        selected["reason"] = f"selected best quantity candidate; score={selected.get('score', 0)}"

    return selected


def build_api_candidate_metadata(entity: Dict[str, Any]) -> Tuple[str, str]:
    country_labels = []
    instance_labels = []

    try:
        country_labels = labels_for_claim(entity, "P17")
    except Exception:
        country_labels = []

    try:
        instance_labels = labels_for_claim(entity, "P31")
    except Exception:
        instance_labels = []

    return (
        ", ".join(country_labels),
        ", ".join(instance_labels),
    )


def build_api_candidate_description(
    entity: Dict[str, Any],
    search_item: Optional[Dict[str, Any]] = None,
) -> str:
    descriptions = [
        entity_description(entity),
    ]

    if search_item:
        descriptions.append(search_item.get("description", ""))

    return build_search_text(*descriptions)


def resolve_place_coordinates_by_sparql(
    place_name: str,
    context: str = "",
) -> Dict[str, Any]:
    label_values = sparql_label_values([clean_text(place_name)])

    query = f"""
SELECT ?item ?itemLabel ?coord ?countryLabel ?instanceLabel WHERE {{
  VALUES ?lookupLabel {{ {label_values} }}

  ?item rdfs:label ?lookupLabel.
  ?item wdt:P625 ?coord.

  BIND(STR(?lookupLabel) AS ?itemLabel)

  OPTIONAL {{
    ?item wdt:P17 ?country.
    ?country rdfs:label ?countryLabel.
    FILTER(LANG(?countryLabel) = "en")
  }}

  OPTIONAL {{
    ?item wdt:P31 ?instance.
    ?instance rdfs:label ?instanceLabel.
    FILTER(LANG(?instanceLabel) = "en")
  }}
}}
LIMIT 50
""".strip()

    data = query_wikidata_sparql(query)
    bindings = data.get("results", {}).get("bindings", [])
    candidates = []

    for binding in bindings:
        item_uri = binding.get("item", {}).get("value", "")
        item_label = binding.get("itemLabel", {}).get("value", "")
        country_label = binding.get("countryLabel", {}).get("value", "")
        instance_label = binding.get("instanceLabel", {}).get("value", "")
        coord_value = binding.get("coord", {}).get("value", "")

        coordinates = parse_wikidata_point(coord_value)

        if not item_uri or not coordinates:
            continue

        qid = qid_from_uri(item_uri)

        candidates.append(
            {
                "qid": qid,
                "label": item_label,
                "country": country_label,
                "instance": instance_label,
                "description": "",
                "coordinates": coordinates,
                "source": entity_url(qid),
                "method": "wikidata_sparql_exact_label",
            }
        )

    selected = select_place_candidate(
        candidates=candidates,
        place_name=place_name,
        context=context,
    )

    if not selected:
        return empty_lookup_result(
            value_key="coordinates",
            input_name=place_name,
            context=context,
            reason="SPARQL found no acceptable country-matching place candidate",
            candidates=candidates,
        )

    return {
        "ok": True,
        "coordinates": selected["coordinates"],
        "source": selected["source"],
        "status": "wikidata",
        "method": selected["method"],
        "qid": selected["qid"],
        "label": selected["label"],
        "country": selected["country"],
        "reason": selected["reason"],
        "candidates": candidates,
        "input": place_name,
        "context": context,
    }


def resolve_place_coordinates_by_api_search(
    place_name: str,
    context: str = "",
    context_suffix: str = "",
) -> Dict[str, Any]:
    search_queries = unique_texts(
        [
            build_search_text(place_name, context),
            build_search_text(place_name, context, context_suffix),
            build_search_text(place_name, context_suffix),
            build_search_text(place_name),
        ]
    )

    candidates = []

    for query in search_queries:
        try:
            search_results = search_wikidata_entities(query, limit=20)
        except Exception as error:
            candidates.append(
                {
                    "qid": "",
                    "label": place_name,
                    "country": context,
                    "instance": "",
                    "description": "",
                    "coordinates": None,
                    "source": "",
                    "method": f"wikidata_api_search: {query}",
                    "decision": "error",
                    "reason": str(error),
                }
            )
            continue

        qids = [item["id"] for item in search_results if item.get("id")]

        try:
            entities = get_wikidata_entities(qids)
        except Exception as error:
            candidates.append(
                {
                    "qid": "",
                    "label": place_name,
                    "country": context,
                    "instance": "",
                    "description": "",
                    "coordinates": None,
                    "source": "",
                    "method": f"wikidata_api_entities: {query}",
                    "decision": "error",
                    "reason": str(error),
                }
            )
            continue

        search_items_by_qid = {
            item["id"]: item
            for item in search_results
            if item.get("id")
        }

        for qid in qids:
            entity = entities.get(qid, {})
            search_item = search_items_by_qid.get(qid, {})
            coordinates = extract_coordinates(entity)

            if not coordinates:
                continue

            country_label, instance_label = build_api_candidate_metadata(entity)
            description = build_api_candidate_description(entity, search_item)

            candidates.append(
                {
                    "qid": qid,
                    "label": entity_label(entity) or search_item.get("label", ""),
                    "country": country_label,
                    "instance": instance_label,
                    "description": description,
                    "coordinates": coordinates,
                    "source": entity_url(qid),
                    "method": f"wikidata_api_search: {query}",
                }
            )

    selected = select_place_candidate(
        candidates=candidates,
        place_name=place_name,
        context=context,
    )

    if not selected:
        return empty_lookup_result(
            value_key="coordinates",
            input_name=place_name,
            context=context,
            reason="API search found no acceptable country-matching place candidate",
            candidates=candidates,
        )

    return {
        "ok": True,
        "coordinates": selected["coordinates"],
        "source": selected["source"],
        "status": "wikidata",
        "method": selected["method"],
        "qid": selected["qid"],
        "label": selected["label"],
        "country": selected.get("country", ""),
        "reason": selected["reason"],
        "candidates": candidates,
        "input": place_name,
        "context": context,
    }


def resolve_place_coordinates(
    place_name: str,
    context: str = "",
    context_suffix: str = "",
) -> Dict[str, Any]:
    all_candidates = []

    try:
        sparql_result = resolve_place_coordinates_by_sparql(
            place_name=place_name,
            context=context,
        )

        all_candidates.extend(sparql_result.get("candidates", []))

        if sparql_result["ok"]:
            return sparql_result
    except Exception as error:
        all_candidates.append(
            {
                "qid": "",
                "label": place_name,
                "country": context,
                "instance": "",
                "description": "",
                "coordinates": None,
                "source": "",
                "method": "wikidata_sparql_exact_label",
                "decision": "error",
                "reason": f"SPARQL failed, continuing with API fallback: {error}",
            }
        )

    api_result = resolve_place_coordinates_by_api_search(
        place_name=place_name,
        context=context,
        context_suffix=context_suffix,
    )

    all_candidates.extend(api_result.get("candidates", []))

    if api_result["ok"]:
        api_result["candidates"] = all_candidates
        return api_result

    return empty_lookup_result(
        value_key="coordinates",
        input_name=place_name,
        context=context,
        reason="No Wikidata coordinate candidate matched the requested context",
        candidates=all_candidates,
    )


def resolve_wikidata_quantity_by_sparql(
    entity_name: str,
    property_id: str,
    context: str = "",
    context_suffix: str = "",
) -> Dict[str, Any]:
    label_variants = generate_label_variants(entity_name, context_suffix)
    label_values = sparql_label_values(label_variants)

    query = f"""
SELECT ?item ?itemLabel ?value ?countryLabel ?instanceLabel WHERE {{
  VALUES ?lookupLabel {{ {label_values} }}

  ?item rdfs:label ?lookupLabel.
  ?item wdt:{property_id} ?value.

  BIND(STR(?lookupLabel) AS ?itemLabel)

  OPTIONAL {{
    ?item wdt:P17 ?country.
    ?country rdfs:label ?countryLabel.
    FILTER(LANG(?countryLabel) = "en")
  }}

  OPTIONAL {{
    ?item wdt:P31 ?instance.
    ?instance rdfs:label ?instanceLabel.
    FILTER(LANG(?instanceLabel) = "en")
  }}
}}
LIMIT 50
""".strip()

    data = query_wikidata_sparql(query)
    bindings = data.get("results", {}).get("bindings", [])
    candidates = []

    for binding in bindings:
        item_uri = binding.get("item", {}).get("value", "")
        item_label = binding.get("itemLabel", {}).get("value", "")
        country_label = binding.get("countryLabel", {}).get("value", "")
        instance_label = binding.get("instanceLabel", {}).get("value", "")
        raw_value = binding.get("value", {}).get("value")

        if not item_uri or raw_value is None:
            continue

        try:
            numeric_value = int(round(float(raw_value)))
        except ValueError:
            continue

        qid = qid_from_uri(item_uri)

        candidates.append(
            {
                "qid": qid,
                "label": item_label,
                "country": country_label,
                "instance": instance_label,
                "description": "",
                "value": numeric_value,
                "source": entity_url(qid),
                "method": "wikidata_sparql_exact_label",
            }
        )

    selected = select_quantity_candidate(
        candidates=candidates,
        entity_name=entity_name,
        context=context,
        property_id=property_id,
    )

    if not selected:
        return empty_lookup_result(
            value_key="value",
            input_name=entity_name,
            context=context,
            reason="SPARQL found no acceptable quantity candidate",
            candidates=candidates,
        )

    return {
        "ok": True,
        "value": selected["value"],
        "source": selected["source"],
        "status": "wikidata",
        "method": selected["method"],
        "qid": selected["qid"],
        "label": selected["label"],
        "country": selected["country"],
        "reason": selected["reason"],
        "candidates": candidates,
        "input": entity_name,
        "context": context,
    }


def resolve_wikidata_quantity_by_api_search(
    entity_name: str,
    context: str,
    property_id: str,
    context_suffix: str = "",
) -> Dict[str, Any]:
    search_queries = unique_texts(
        [
            build_search_text(entity_name, context, context_suffix),
            build_search_text(entity_name, context),
            build_search_text(entity_name, context_suffix),
            build_search_text(entity_name),
        ]
    )

    candidates = []

    for query in search_queries:
        try:
            search_results = search_wikidata_entities(query, limit=20)
        except Exception as error:
            candidates.append(
                {
                    "qid": "",
                    "label": entity_name,
                    "country": context,
                    "instance": "",
                    "description": "",
                    "value": None,
                    "source": "",
                    "method": f"wikidata_api_search: {query}",
                    "decision": "error",
                    "reason": str(error),
                }
            )
            continue

        qids = [item["id"] for item in search_results if item.get("id")]

        try:
            entities = get_wikidata_entities(qids)
        except Exception as error:
            candidates.append(
                {
                    "qid": "",
                    "label": entity_name,
                    "country": context,
                    "instance": "",
                    "description": "",
                    "value": None,
                    "source": "",
                    "method": f"wikidata_api_entities: {query}",
                    "decision": "error",
                    "reason": str(error),
                }
            )
            continue

        search_items_by_qid = {
            item["id"]: item
            for item in search_results
            if item.get("id")
        }

        for qid in qids:
            entity = entities.get(qid, {})
            search_item = search_items_by_qid.get(qid, {})
            value = extract_quantity_amount(entity, property_id)

            if value is None:
                continue

            country_label, instance_label = build_api_candidate_metadata(entity)
            description = build_api_candidate_description(entity, search_item)

            candidates.append(
                {
                    "qid": qid,
                    "label": entity_label(entity) or search_item.get("label", ""),
                    "country": country_label,
                    "instance": instance_label,
                    "description": description,
                    "value": value,
                    "source": entity_url(qid),
                    "method": f"wikidata_api_search: {query}",
                }
            )

    selected = select_quantity_candidate(
        candidates=candidates,
        entity_name=entity_name,
        context=context,
        property_id=property_id,
    )

    if not selected:
        return empty_lookup_result(
            value_key="value",
            input_name=entity_name,
            context=context,
            reason="API search found no acceptable quantity candidate",
            candidates=candidates,
        )

    return {
        "ok": True,
        "value": selected["value"],
        "source": selected["source"],
        "status": "wikidata",
        "method": selected["method"],
        "qid": selected["qid"],
        "label": selected["label"],
        "country": selected.get("country", ""),
        "reason": selected["reason"],
        "candidates": candidates,
        "input": entity_name,
        "context": context,
    }


def resolve_wikidata_quantity_property(
    entity_name: str,
    context: str,
    property_id: str,
    context_suffix: str = "",
) -> Dict[str, Any]:
    all_candidates = []

    try:
        sparql_result = resolve_wikidata_quantity_by_sparql(
            entity_name=entity_name,
            property_id=property_id,
            context=context,
            context_suffix=context_suffix,
        )

        all_candidates.extend(sparql_result.get("candidates", []))

        if sparql_result["ok"]:
            return sparql_result
    except Exception as error:
        all_candidates.append(
            {
                "qid": "",
                "label": entity_name,
                "country": context,
                "instance": "",
                "description": "",
                "value": None,
                "source": "",
                "method": "wikidata_sparql_exact_label",
                "decision": "error",
                "reason": f"SPARQL failed, continuing with API fallback: {error}",
            }
        )

    api_result = resolve_wikidata_quantity_by_api_search(
        entity_name=entity_name,
        context=context,
        property_id=property_id,
        context_suffix=context_suffix,
    )

    all_candidates.extend(api_result.get("candidates", []))

    if api_result["ok"]:
        api_result["candidates"] = all_candidates
        return api_result

    return empty_lookup_result(
        value_key="value",
        input_name=entity_name,
        context=context,
        reason="No Wikidata quantity candidate matched the requested context",
        candidates=all_candidates,
    )

In [69]:
# Chunk 8: Excel processing, logging, retrying, and saving functions
def is_empty_value(value: Any) -> bool:
    if pd.isna(value):
        return True

    return str(value).strip() == ""


def format_excel_file(file_path: str) -> None:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    workbook = load_workbook(file_path)
    worksheet = workbook.active

    header_fill = PatternFill("solid", fgColor="EAF2F8")

    for cell in worksheet[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")

    worksheet.freeze_panes = "A2"

    for column_cells in worksheet.columns:
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in column_cells
        )
        column_letter = get_column_letter(column_cells[0].column)
        worksheet.column_dimensions[column_letter].width = min(max(max_length + 2, 12), 45)

    workbook.save(file_path)


def build_context_from_columns(row: pd.Series, columns: List[str]) -> str:
    return build_search_text(*(row[column] for column in columns))


def build_place_lookup_args(
    row: pd.Series,
    place_columns: List[str],
) -> Tuple[str, str]:
    place_name = clean_text(row[place_columns[0]])
    context = build_context_from_columns(row, place_columns[1:])
    return place_name, context


def lookup_result_is_valid(result: Optional[Dict[str, Any]], value_key: str) -> bool:
    if not result or not result.get("ok"):
        return False

    value = result.get(value_key)

    if value_key == "coordinates":
        return (
            isinstance(value, (tuple, list))
            and len(value) == 2
            and value[0] is not None
            and value[1] is not None
        )

    return value is not None


def retry_unresolved_lookup_results(
    lookup_results: Dict[Tuple[str, str], Dict[str, Any]],
    value_key: str,
    resolver: Callable[..., Dict[str, Any]],
    build_resolver_kwargs: Callable[[Tuple[str, str]], Dict[str, Any]],
    attempts: int = 3,
    delay_seconds: int = 2,
    description: str = "Retrying unresolved lookups",
) -> Dict[Tuple[str, str], Dict[str, Any]]:
    import time

    unresolved_keys = [
        key for key, result in lookup_results.items()
        if not lookup_result_is_valid(result, value_key)
    ]

    if not unresolved_keys:
        return lookup_results

    for key in tqdm(unresolved_keys, desc=description):
        previous_result = lookup_results.get(key, {})
        accumulated_candidates = list(previous_result.get("candidates", []))
        last_result = previous_result

        for attempt in range(1, attempts + 1):
            if attempt > 1:
                time.sleep(delay_seconds * attempt)

            try:
                result = resolver(**build_resolver_kwargs(key))
                result_candidates = result.get("candidates", [])
                accumulated_candidates.extend(result_candidates)
                result["candidates"] = accumulated_candidates

                if lookup_result_is_valid(result, value_key):
                    original_reason = result.get("reason", "")
                    result["reason"] = f"resolved on retry {attempt}; {original_reason}"
                    lookup_results[key] = result
                    break

                last_result = result
            except Exception as error:
                accumulated_candidates.append(
                    {
                        "qid": "",
                        "label": key[0],
                        "country": key[1],
                        "instance": "",
                        "description": "",
                        value_key: None,
                        "source": "",
                        "method": "retry",
                        "decision": "error",
                        "reason": str(error),
                    }
                )

                last_result = empty_lookup_result(
                    value_key=value_key,
                    input_name=key[0],
                    context=key[1],
                    reason=f"retry {attempt} failed: {error}",
                    candidates=accumulated_candidates,
                )

        if not lookup_result_is_valid(lookup_results.get(key), value_key):
            last_result["candidates"] = accumulated_candidates
            lookup_results[key] = last_result

    return lookup_results


def format_candidate_log(result: Dict[str, Any], max_items: int = 6) -> str:
    candidates = result.get("candidates", [])

    if not candidates:
        return ""

    return " | ".join(
        candidate_log_entry(candidate)
        for candidate in candidates[:max_items]
    )


def build_lookup_audit_frame(
    lookup_results: Dict[Tuple[str, str], Dict[str, Any]],
    value_key: str,
) -> pd.DataFrame:
    rows = []

    for key, result in sorted(lookup_results.items()):
        input_name, context = key

        rows.append(
            {
                "input": input_name,
                "context": context,
                "ok": result.get("ok"),
                "status": result.get("status"),
                "method": result.get("method"),
                "qid": result.get("qid"),
                "label": result.get("label"),
                "country": result.get("country"),
                value_key: result.get(value_key),
                "source": result.get("source"),
                "reason": result.get("reason"),
                "candidate_log": format_candidate_log(result),
            }
        )

    return pd.DataFrame(rows)


def display_lookup_audit(
    title: str,
    lookup_results: Dict[Tuple[str, str], Dict[str, Any]],
    value_key: str,
) -> None:
    audit_df = build_lookup_audit_frame(
        lookup_results=lookup_results,
        value_key=value_key,
    )

    display(Markdown(f"### {title}"))
    display(audit_df)


def build_distance_row_audit(
    df: pd.DataFrame,
    target_column: str,
    from_place_columns: List[str],
    to_place_columns: List[str],
    coordinate_results: Dict[Tuple[str, str], Dict[str, Any]],
) -> pd.DataFrame:
    rows = []

    for index, row in df.iterrows():
        from_key = build_place_lookup_args(row, from_place_columns)
        to_key = build_place_lookup_args(row, to_place_columns)

        from_result = coordinate_results.get(from_key, {})
        to_result = coordinate_results.get(to_key, {})

        rows.append(
            {
                "row": index,
                "from_input": from_key[0],
                "from_context": from_key[1],
                "from_qid": from_result.get("qid"),
                "from_label": from_result.get("label"),
                "from_country": from_result.get("country"),
                "from_coordinates": from_result.get("coordinates"),
                "from_reason": from_result.get("reason"),
                "to_input": to_key[0],
                "to_context": to_key[1],
                "to_qid": to_result.get("qid"),
                "to_label": to_result.get("label"),
                "to_country": to_result.get("country"),
                "to_coordinates": to_result.get("coordinates"),
                "to_reason": to_result.get("reason"),
                target_column: row.get(target_column),
            }
        )

    return pd.DataFrame(rows)


def display_distance_row_audit(
    df: pd.DataFrame,
    target_column: str,
    from_place_columns: List[str],
    to_place_columns: List[str],
    coordinate_results: Dict[Tuple[str, str], Dict[str, Any]],
) -> None:
    audit_df = build_distance_row_audit(
        df=df,
        target_column=target_column,
        from_place_columns=from_place_columns,
        to_place_columns=to_place_columns,
        coordinate_results=coordinate_results,
    )

    display(Markdown("### Distance row audit"))
    display(audit_df)


def enrich_place_distances(
    df: pd.DataFrame,
    target_column: str,
    from_place_columns: List[str],
    to_place_columns: List[str],
    required_columns: List[str],
    place_context_suffix: str,
    overwrite: bool,
    include_debug_columns: bool,
    max_workers: int,
    show_lookup_log: bool = True,
) -> pd.DataFrame:
    validate_required_columns(df, required_columns)

    if target_column not in df.columns:
        df[target_column] = None

    source_column = f"{target_column}_source"
    status_column = f"{target_column}_status"
    qid_column = f"{target_column}_qid"
    method_column = f"{target_column}_method"
    reason_column = f"{target_column}_reason"

    if include_debug_columns:
        df[source_column] = ""
        df[status_column] = ""
        df[qid_column] = ""
        df[method_column] = ""
        df[reason_column] = ""

    unique_places = set()

    for _, row in df.iterrows():
        unique_places.add(build_place_lookup_args(row, from_place_columns))
        unique_places.add(build_place_lookup_args(row, to_place_columns))

    coordinate_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(
                resolve_place_coordinates,
                place_name,
                context,
                place_context_suffix,
            ): (place_name, context)
            for place_name, context in unique_places
        }

        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Resolving places"):
            key = future_map[future]

            try:
                coordinate_results[key] = future.result()
            except Exception as error:
                coordinate_results[key] = empty_lookup_result(
                    value_key="coordinates",
                    input_name=key[0],
                    context=key[1],
                    reason=f"resolver error: {error}",
                )

    coordinate_results = retry_unresolved_lookup_results(
        lookup_results=coordinate_results,
        value_key="coordinates",
        resolver=resolve_place_coordinates,
        build_resolver_kwargs=lambda key: {
            "place_name": key[0],
            "context": key[1],
            "context_suffix": place_context_suffix,
        },
        attempts=3,
        delay_seconds=2,
        description="Retrying unresolved places",
    )

    if show_lookup_log:
        display_lookup_audit(
            title="Place lookup audit",
            lookup_results=coordinate_results,
            value_key="coordinates",
        )

    for index, row in df.iterrows():
        current_value = row.get(target_column)

        if not overwrite and not is_empty_value(current_value):
            continue

        from_key = build_place_lookup_args(row, from_place_columns)
        to_key = build_place_lookup_args(row, to_place_columns)

        from_result = coordinate_results.get(from_key)
        to_result = coordinate_results.get(to_key)

        if (
            not lookup_result_is_valid(from_result, "coordinates")
            or not lookup_result_is_valid(to_result, "coordinates")
        ):
            df.at[index, target_column] = None

            if include_debug_columns:
                df.at[index, source_column] = ""
                df.at[index, status_column] = "not_found"
                df.at[index, qid_column] = ""
                df.at[index, method_column] = ""
                df.at[index, reason_column] = build_search_text(
                    from_result.get("reason") if from_result else "",
                    to_result.get("reason") if to_result else "",
                )

            continue

        lat1, lon1 = from_result["coordinates"]
        lat2, lon2 = to_result["coordinates"]

        distance_km = round(haversine_km(lat1, lon1, lat2, lon2))
        df.at[index, target_column] = distance_km

        if include_debug_columns:
            df.at[index, source_column] = f"{from_result['source']} | {to_result['source']}"
            df.at[index, status_column] = "wikidata"
            df.at[index, qid_column] = f"{from_result['qid']} | {to_result['qid']}"
            df.at[index, method_column] = f"{from_result['method']} | {to_result['method']}"
            df.at[index, reason_column] = f"{from_result['reason']} | {to_result['reason']}"

    if show_lookup_log:
        display_distance_row_audit(
            df=df,
            target_column=target_column,
            from_place_columns=from_place_columns,
            to_place_columns=to_place_columns,
            coordinate_results=coordinate_results,
        )

    return df


def enrich_wikidata_quantity_property(
    df: pd.DataFrame,
    target_column: str,
    entity_column: str,
    context_columns: List[str],
    required_columns: List[str],
    wikidata_property_id: str,
    entity_context_suffix: str,
    overwrite: bool,
    include_debug_columns: bool,
    max_workers: int,
    show_lookup_log: bool = True,
) -> pd.DataFrame:
    validate_required_columns(df, required_columns)

    if target_column not in df.columns:
        df[target_column] = None

    source_column = f"{target_column}_source"
    status_column = f"{target_column}_status"
    qid_column = f"{target_column}_qid"
    method_column = f"{target_column}_method"
    reason_column = f"{target_column}_reason"

    if include_debug_columns:
        df[source_column] = ""
        df[status_column] = ""
        df[qid_column] = ""
        df[method_column] = ""
        df[reason_column] = ""

    unique_entities = {
        (
            clean_text(row[entity_column]),
            build_context_from_columns(row, context_columns),
        )
        for _, row in df.iterrows()
    }

    quantity_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(
                resolve_wikidata_quantity_property,
                entity_name,
                context,
                wikidata_property_id,
                entity_context_suffix,
            ): (entity_name, context)
            for entity_name, context in unique_entities
        }

        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Resolving Wikidata values"):
            key = future_map[future]

            try:
                quantity_results[key] = future.result()
            except Exception as error:
                quantity_results[key] = empty_lookup_result(
                    value_key="value",
                    input_name=key[0],
                    context=key[1],
                    reason=f"resolver error: {error}",
                )

    quantity_results = retry_unresolved_lookup_results(
        lookup_results=quantity_results,
        value_key="value",
        resolver=resolve_wikidata_quantity_property,
        build_resolver_kwargs=lambda key: {
            "entity_name": key[0],
            "context": key[1],
            "property_id": wikidata_property_id,
            "context_suffix": entity_context_suffix,
        },
        attempts=3,
        delay_seconds=2,
        description="Retrying unresolved Wikidata values",
    )

    if show_lookup_log:
        display_lookup_audit(
            title="Quantity lookup audit",
            lookup_results=quantity_results,
            value_key="value",
        )

    for index, row in df.iterrows():
        current_value = row.get(target_column)

        if not overwrite and not is_empty_value(current_value):
            continue

        key = (
            clean_text(row[entity_column]),
            build_context_from_columns(row, context_columns),
        )
        result = quantity_results.get(key)

        if not lookup_result_is_valid(result, "value"):
            df.at[index, target_column] = None

            if include_debug_columns:
                df.at[index, source_column] = ""
                df.at[index, status_column] = "not_found"
                df.at[index, qid_column] = ""
                df.at[index, method_column] = ""
                df.at[index, reason_column] = result.get("reason") if result else "missing lookup result"

            continue

        df.at[index, target_column] = result["value"]

        if include_debug_columns:
            df.at[index, source_column] = result["source"]
            df.at[index, status_column] = result["status"]
            df.at[index, qid_column] = result["qid"]
            df.at[index, method_column] = result["method"]
            df.at[index, reason_column] = result["reason"]

    return df


def process_excel(
    file_path: str,
    task_description: str,
    output_path: Optional[str] = None,
    overwrite: bool = True,
    include_debug_columns: bool = True,
    max_workers: int = 4,
    show_lookup_log: bool = True,
) -> str:
    df = pd.read_excel(file_path)
    plan = analyze_task(df, task_description)

    print("Enrichment plan:")
    print(json.dumps(plan, ensure_ascii=False, indent=2))

    task_type = plan["task_type"]
    target_column = plan["target_column"]

    if task_type == "unsupported":
        raise ValueError("Unsupported enrichment task.")

    if task_type == "distance_between_places":
        enriched_df = enrich_place_distances(
            df=df,
            target_column=target_column,
            from_place_columns=plan["from_place_columns"],
            to_place_columns=plan["to_place_columns"],
            required_columns=plan["required_columns"],
            place_context_suffix=plan.get("place_context_suffix", ""),
            overwrite=overwrite,
            include_debug_columns=include_debug_columns,
            max_workers=max_workers,
            show_lookup_log=show_lookup_log,
        )
    elif task_type == "wikidata_quantity_property":
        enriched_df = enrich_wikidata_quantity_property(
            df=df,
            target_column=target_column,
            entity_column=plan["entity_column"],
            context_columns=plan.get("context_columns", []),
            required_columns=plan["required_columns"],
            wikidata_property_id=plan["wikidata_property_id"],
            entity_context_suffix=plan.get("entity_context_suffix", ""),
            overwrite=overwrite,
            include_debug_columns=include_debug_columns,
            max_workers=max_workers,
            show_lookup_log=show_lookup_log,
        )
    else:
        raise ValueError(f"Unsupported task type: {task_type}")

    if output_path is None:
        input_path = Path(file_path)
        output_path = str(input_path.with_name(f"{input_path.stem}_enriched.xlsx"))

    enriched_df.to_excel(output_path, index=False)
    format_excel_file(output_path)

    print(f"Saved: {output_path}")
    return output_path

In [71]:
# Chunk 9: Download correct input files and run enrichment
capitals_url = "https://docs.google.com/spreadsheets/d/1Bv_WFNzGG4ZL7-QQf8dSrnSEwRvUnhrh/edit?gid=28170114#gid=28170114"
mountains_url = "https://docs.google.com/spreadsheets/d/1UutRbt0yvinVpz3cQdAcRSlqPFmyVEXx/edit?gid=1262408660#gid=1262408660"

download_google_sheet_as_xlsx(capitals_url, "capitals.xlsx")
download_google_sheet_as_xlsx(mountains_url, "mountains.xlsx")

capitals_output = process_excel(
    file_path="capitals.xlsx",
    task_description="знайди пряму відстань між столицями в км для колонки distance",
    output_path="capitals_enriched.xlsx",
    include_debug_columns=False,
    max_workers=2,
    show_lookup_log=False,
)

mountains_output = process_excel(
    file_path="mountains.xlsx",
    task_description="знайди висоту гір у метрах для колонки height",
    output_path="mountains_enriched.xlsx",
    include_debug_columns=False,
    max_workers=2,
    show_lookup_log=False,
)

print(capitals_output)
print(mountains_output)

Enrichment plan:
{
  "task_type": "distance_between_places",
  "target_column": "distance",
  "required_columns": [
    "Capital_From",
    "Country_From",
    "Capital_To",
    "Country_To"
  ],
  "from_place_columns": [
    "Capital_From",
    "Country_From"
  ],
  "to_place_columns": [
    "Capital_To",
    "Country_To"
  ],
  "place_context_suffix": "capital city",
  "value_unit": "km"
}


Resolving places:   0%|          | 0/14 [00:00<?, ?it/s]

Retrying unresolved places:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: capitals_enriched.xlsx
Enrichment plan:
{
  "task_type": "wikidata_quantity_property",
  "target_column": "height",
  "required_columns": [
    "Mountain"
  ],
  "entity_column": "Mountain",
  "context_columns": [
    "Country"
  ],
  "entity_context_suffix": "mountain",
  "wikidata_property_id": "P2044",
  "value_unit": "m"
}


Resolving Wikidata values:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: mountains_enriched.xlsx
capitals_enriched.xlsx
mountains_enriched.xlsx


In [72]:
# Chunk 10: Preview and validate enriched Excel data
capitals_result = pd.read_excel("capitals_enriched.xlsx")
mountains_result = pd.read_excel("mountains_enriched.xlsx")

display(Markdown("## Capitals enriched"))
display(capitals_result)

display(Markdown("## Mountains enriched"))
display(mountains_result)

assert "distance_source" not in capitals_result.columns
assert "distance_status" not in capitals_result.columns
assert "height_source" not in mountains_result.columns
assert "height_status" not in mountains_result.columns

assert "distance" in capitals_result.columns
assert "height" in mountains_result.columns

assert capitals_result["distance"].notna().all()
assert mountains_result["height"].notna().all()

print("Validation passed: original structure preserved and target columns filled.")

## Capitals enriched

,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,2023
1,London,United Kingdom,Rome,Italy,1434
2,Paris,France,Berlin,Germany,876
3,Berlin,Germany,Vienna,Austria,524
4,Warsaw,Poland,Kyiv,Ukraine,689
5,Rome,Italy,Madrid,Spain,1358
6,Madrid,Spain,Lisbon,Portugal,509
7,Vienna,Austria,Budapest,Hungary,214
8,Stockholm,Sweden,Oslo,Norway,417
9,Athens,Greece,Istanbul,Turkey,561


## Mountains enriched

,Mountain,Country,height
0,Mount Everest,Nepal/China,8849
1,K2,Pakistan/China,8611
2,Kangchenjunga,Nepal/India,8586
3,Lhotse,Nepal/China,8516
4,Makalu,Nepal/China,8485
5,Cho Oyu,Nepal/China,8188
6,Dhaulagiri,Nepal,8167
7,Manaslu,Nepal,8163
8,Nanga Parbat,Pakistan,8126
9,Annapurna,Nepal,8091


Validation passed: original structure preserved and target columns filled.


In [74]:
# Chunk 11: Download enriched files
download_result_file("capitals_enriched.xlsx")
download_result_file("mountains_enriched.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>